# Gait Distraction-Measure Dataset Builder

**Goal of the wider project:** predict *which distraction measure* (experimental
scenario) a subject was under, from the features of their gait.

In `Cleaned Data/` each subject has four CSVs whose **filename is the condition**.
The four codes are the paper's four experimental scenarios (Alamdar et al., *The
cognitive-motor coordination dataset*, Sci Data 2026, s41597-026-08103-4) and
decode as `<vision><feedback>`:

| Code | Vision (`BF` / `CL`) | Vibrotactile feedback (`NF` / `WF`) |
|---|---|---|
| `BFNF` | Blindfold (closed-eye) | No feedback |
| `BFWF` | Blindfold (closed-eye) | With feedback |
| `CLNF` | VR-induced cognitive load | No feedback |
| `CLWF` | VR-induced cognitive load | With feedback |

So the two distraction factors are **vision** (blindfold vs. VR cognitive load)
and whether **vibrotactile feedback** was provided. Every row of a file is one
200 Hz time sample with ~28 sensor channels (force plates, treadmill speed, foot
sensor, camera, vibro-tactor states, task scores, VR pose).

This notebook does two things:

1. **Concatenate** every file in `Cleaned Data/` into one long CSV, adding a
   `subject_id` feature and a `condition` **target** column (parsed from the
   filename), plus human-readable `vision` / `feedback` / `condition_label`
   columns. Written incrementally to disk so it never has to hold all ~9 GB in
   memory at once.
2. **Split** the result into train / validation / test sets that each hold the
   **same proportion of every subject** (stratified on `subject_id`).

In [ ]:
import csv
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# --- Configuration -------------------------------------------------------
# Paths are relative to this notebook's folder (the repo root), so this runs
# unchanged on any machine that has `Cleaned Data/` alongside the notebook.
CLEANED_DIR = Path("Cleaned Data")
OUT_DIR = Path("dataset")
OUT_DIR.mkdir(exist_ok=True)

COMBINED_CSV = OUT_DIR / "combined_gait.csv"

# The four condition codes (= CSV filenames) decode as <vision><feedback>.
# Source: Alamdar et al., "The cognitive-motor coordination dataset",
# Sci Data 2026 (s41597-026-08103-4) -- four experimental scenarios.
VISION = {"BF": "blindfold", "CL": "vr_cognitive_load"}
FEEDBACK = {"NF": "no_feedback", "WF": "with_feedback"}
CONDITION_LABELS = {
    "BFNF": "blindfold_no_feedback",
    "BFWF": "blindfold_with_feedback",
    "CLNF": "vr_cognitive_load_no_feedback",
    "CLWF": "vr_cognitive_load_with_feedback",
}

# Split fractions (must sum to 1.0). Val + test are carved out of the whole.
TEST_FRAC = 0.15
VAL_FRAC = 0.15
TRAIN_FRAC = 1.0 - TEST_FRAC - VAL_FRAC
RANDOM_STATE = 42

assert CLEANED_DIR.is_dir(), f"Cannot find {CLEANED_DIR.resolve()}"

## 1. Concatenate `Cleaned Data/` into one CSV

We walk `Cleaned Data/sub*/` and stream every row of every condition file into
a single output CSV, prepending id + target columns:

* `subject_id` &mdash; the folder name (e.g. `sub01`), a **feature**.
* `condition` &mdash; the filename stem (`BFNF` / `BFWF` / `CLNF` / `CLWF`); the
  compact **target** for the ML task.
* `vision` &mdash; `blindfold` or `vr_cognitive_load` (the `BF`/`CL` factor).
* `feedback` &mdash; `no_feedback` or `with_feedback` (the `NF`/`WF` factor).
* `condition_label` &mdash; the full human-readable scenario name.

Streaming row-by-row (rather than `pd.concat` of ~112 DataFrames) keeps the peak
memory tiny, so the full multi-GB dataset builds even on a laptop.

In [ ]:
condition_files = sorted(CLEANED_DIR.glob("sub*/*.csv"))
print(f"Found {len(condition_files)} condition files across "
      f"{len({p.parent.name for p in condition_files})} subjects.")

# Sanity-check that every file shares the same header, so a single stitched
# header is valid for the combined CSV.
def read_header(path):
    with open(path, newline="") as fh:
        return next(csv.reader(fh))

headers = {tuple(read_header(p)) for p in condition_files}
assert len(headers) == 1, f"Files disagree on columns: {len(headers)} distinct headers"
sensor_cols = list(next(iter(headers)))
print(f"{len(sensor_cols)} sensor columns per file:\n{sensor_cols}")

In [ ]:
# Stream every file into one CSV, prepending the id + target columns:
#   subject_id, condition (code), vision, feedback, condition_label
out_header = ["subject_id", "condition", "vision", "feedback",
              "condition_label", *sensor_cols]
total_rows = 0

with open(COMBINED_CSV, "w", newline="") as out_fh:
    writer = csv.writer(out_fh)
    writer.writerow(out_header)

    for path in condition_files:
        subject_id = path.parent.name      # e.g. "sub01"
        condition = path.stem              # e.g. "BFNF" (the target label)
        assert condition in CONDITION_LABELS, f"Unexpected condition: {condition}"
        vision = VISION[condition[:2]]
        feedback = FEEDBACK[condition[2:]]
        label = CONDITION_LABELS[condition]
        prefix = [subject_id, condition, vision, feedback, label]
        with open(path, newline="") as in_fh:
            reader = csv.reader(in_fh)
            next(reader)                   # skip the per-file header
            file_rows = 0
            for row in reader:
                writer.writerow([*prefix, *row])
                file_rows += 1
        total_rows += file_rows
        print(f"  {subject_id}/{condition}: {file_rows:>7,} rows")

print(f"\nWrote {total_rows:,} rows to {COMBINED_CSV} "
      f"({COMBINED_CSV.stat().st_size / 1e9:.2f} GB).")

## 2. Load the combined CSV

We downcast float channels to `float32` to roughly halve the in-memory
footprint. `subject_id` and `condition` are kept as `category`.

In [ ]:
label_cols = ["subject_id", "condition", "vision", "feedback", "condition_label"]
dtype = {c: "float32" for c in sensor_cols}
dtype.update({c: "category" for c in label_cols})

df = pd.read_csv(COMBINED_CSV, dtype=dtype)
print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns")
print("\nRows per condition (target distribution):")
print(df["condition_label"].value_counts())
print("\nRows per subject (head):")
print(df["subject_id"].value_counts().head())
df.head()

## 3. Stratified train / validation / test split

Requirement: **each split must contain the same proportion of every subject.**
We stratify on `subject_id` in two stages:

1. Split off the **test** set (`TEST_FRAC`), stratified by subject.
2. Split the remainder into **train** and **validation**, again stratified by
   subject, sized so val is `VAL_FRAC` of the whole.

> **Note:** stratifying on `subject_id` puts each subject's samples into *all*
> three sets in equal proportion (as requested). Because the same subject then
> appears in train and test, model scores reflect within-subject generalisation,
> not generalisation to unseen people. Swap `stratify=subjects` for a
> `GroupShuffleSplit` on `subject_id` if you later want subject-held-out
> evaluation instead.

In [ ]:
subjects = df["subject_id"]
idx = np.arange(len(df))

# Stage 1: peel off the test set, stratified by subject.
train_val_idx, test_idx = train_test_split(
    idx,
    test_size=TEST_FRAC,
    stratify=subjects,
    random_state=RANDOM_STATE,
)

# Stage 2: split the remainder into train + val, stratified by subject.
# val is VAL_FRAC of the *whole*, so its share of the remainder is rescaled.
val_share_of_remainder = VAL_FRAC / (TRAIN_FRAC + VAL_FRAC)
train_idx, val_idx = train_test_split(
    train_val_idx,
    test_size=val_share_of_remainder,
    stratify=subjects.iloc[train_val_idx],
    random_state=RANDOM_STATE,
)

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

print(f"train: {len(train_df):>9,}  ({len(train_df)/len(df):.1%})")
print(f"val:   {len(val_df):>9,}  ({len(val_df)/len(df):.1%})")
print(f"test:  {len(test_df):>9,}  ({len(test_df)/len(df):.1%})")

In [ ]:
# Verify each split holds the same per-subject proportion.
proportions = pd.DataFrame({
    "train": train_df["subject_id"].value_counts(normalize=True),
    "val": val_df["subject_id"].value_counts(normalize=True),
    "test": test_df["subject_id"].value_counts(normalize=True),
}).sort_index()

print("Per-subject share within each split (should match across columns):")
print(proportions)
max_dev = (proportions.max(axis=1) - proportions.min(axis=1)).max()
print(f"\nMax deviation of any subject's share across splits: {max_dev:.4%}")

In [ ]:
# Persist the splits.
train_df.to_csv(OUT_DIR / "train.csv", index=False)
val_df.to_csv(OUT_DIR / "val.csv", index=False)
test_df.to_csv(OUT_DIR / "test.csv", index=False)
print("Saved train.csv, val.csv, test.csv to", OUT_DIR.resolve())